In [0]:
from pyspark.sql import DataFrame
from delta.tables import DeltaTable 
from pyspark.sql.functions import col, split, trim, lower, when, lpad, length


In [0]:
visit_points = spark.table("dbw_routemind_euskadi_dev.silver.visit_points")


In [0]:
# fix on municipalitycode and territorycode to have same id structure as municipality codes table
visit_localion_norm = visit_points.withColumn(
    "municipalitycode",
    when(length(col("municipalitycode")) < 3, lpad("municipalitycode", 3, "0")).otherwise(col("municipalitycode"))
    ).withColumn("territorycode",
        when(length(col("territorycode")) < 2, lpad("territorycode", 2, "0")).otherwise(col("territorycode"))) 

In [0]:
# apply user categorization to filter in WebApp
visit_points_user_category = visit_localion_norm.withColumn(
    "user_category",
    when(
        (col("templateType") == "Teatros y Cines") | (col("templateType") == "Auditorios  Museos  Galerías de arte y salas de exposiciones"),
        "teatro-arte")
    .when(
        (col("marks") == "Playas"), "playa")
    .when((col("marks") == "Pantanos") | (col("marks") == "Rios") | (col("templateType") == "Espacios Naturales"), "montana")
    .when((col("marks") == "Bodega"), "bodegas")
    .when((col("marks") == "Bodega Txakoli"), "txakoli")
    .when((col("marks") == "Sidreria"), "sidrerias")
    .when((col("marks") == "Trujal") | (col("marks") == "Restaurante"), "gastronomia")
    .when(col("templateType") == "Patrimonio cultural", "monumentos-sitios-historicos")
    .otherwise("eventos-culturales")
)

display(visit_points_user_category.head(20))

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.visit_points_user_category"
delta_path = "abfss://gold@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/visit_points/data"

if not spark.catalog.tableExists(target_table):
    print(f"Table {target_table} doesn't exist. Creating table...")
    
    visit_points_user_category.write \
        .format("delta") \
        .option("path", delta_path) \
        .saveAsTable(target_table)
        
    print(f"table {target_table} created. rows processed: {visit_points_user_category.count()}")

else:
    delta_target = DeltaTable.forName(spark, target_table)
    (
        delta_target.alias("t")
        .merge(
            visit_points_user_category.alias("s"),
            "t.documentName = s.documentName AND t.category = s.category"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE completed on {target_table}. rows processed: {visit_points_user_category.count()}")